In [ ]:
import sys
sys.path.append('../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import torch
from tqdm import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from Network.ResNet19.SEW_Resnet_BF import *
from Network.VGG.VGG11_Net import *

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

In [ ]:
timestep = 6

In [ ]:
dataset_path = '/ssd/Datasets/CIFAR10/'
batch_size = 256
workers = 8
class_number = 10

In [ ]:
transform_train = v2.Compose([
    v2.PILToTensor(),
    v2.Pad(4, padding_mode='reflect'),
    v2.RandomHorizontalFlip(0.5),
    v2.RandomCrop(32),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

transform_test = v2.Compose([
    v2.PILToTensor(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = datasets.CIFAR10(root=dataset_path, train=True, download=True, transform=transform_train)
val_dataset = datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=workers, pin_memory=True)

In [ ]:
model = ResNet19(num_classes=class_number, T=timestep, alpha=0.5)
model_path = f'../models/cifar10_resnet19_t{timestep}.pth.tar'

# model = VGG_11_BF(T = timestep, num_classes=class_number)
# model_path = f'../models/cifar10_vgg11_t{timestep}.pth.tar'

print(f"=> loading checkpoint '{model_path}'")
checkpoint = torch.load(model_path)
model = torch.nn.DataParallel(model).cuda()
functional.set_step_mode(model, step_mode='m')
best_acc1 = checkpoint['best_acc1']
model.load_state_dict(checkpoint['state_dict'])
print(f"=> loaded checkpoint '{model_path}' (epoch {checkpoint['epoch']}) best_acc1: {best_acc1:.3f}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()) / 1000000}")


In [ ]:
Confusion_Matrix = torch.zeros((class_number, class_number))
model.eval()
with torch.no_grad():
    for img, label in tqdm(val_loader):
        img = img.cuda()
        label = label.cuda()
        out_fr = model(img)
        guess = out_fr.argmax(1)
        for j in range(len(label)):
            Confusion_Matrix[label[j],guess[j]] += 1
        functional.reset_net(model)
acc = Confusion_Matrix.diag()
acc = acc.sum()/Confusion_Matrix.sum()
print(f'Confusion_Matrix = \n{Confusion_Matrix}')
print(f'acc = {acc}')